In [1]:
# 01_basic_operations.py
"""
FlagQuantum 入门教程 - 第 1 课：基础门操作
目标：掌握常用的量子门及其效果
"""

import torch

import flagquantum as fq


def tutorial_01_pauli_gates():
    """Pauli 门：X, Y, Z"""
    print("=" * 60)
    print("1.1 Pauli 门")
    print("=" * 60)

    n_wires = 1
    qdev = fq.DistributedQuantumDevice(n_wires=n_wires, bsz=1, device="cpu")

    gates = [
        ("I", fq.I, "恒等门", "|0⟩ → |0⟩, |1⟩ → |1⟩"),
        ("X", fq.X, "NOT门", "|0⟩ → |1⟩, |1⟩ → |0⟩"),
        ("Y", fq.Y, "Y门", "|0⟩ → i|1⟩, |1⟩ → -i|0⟩"),
        ("Z", fq.Z, "Z门", "|0⟩ → |0⟩, |1⟩ → -|1⟩"),
    ]

    for name, gate, desc, effect in gates:
        qdev.reset_states()

        # 先测试 |0⟩ 态
        gate(wires=[0])(qdev)
        states = torch.view_as_complex(qdev.states)

        print(f"\n{name} 门 ({desc}):")
        print(f"  作用在 |0⟩: {states.flatten()}")

        # 再测试 |1⟩ 态
        qdev.reset_states()
        fq.X(wires=[0])(qdev)  # 变成 |1⟩
        gate(wires=[0])(qdev)
        states = torch.view_as_complex(qdev.states)
        print(f"  作用在 |1⟩: {states.flatten()}")
        print(f"  效果: {effect}")

In [2]:
tutorial_01_pauli_gates()

1.1 Pauli 门

I 门 (恒等门):
  作用在 |0⟩: tensor([1.+0.j, 0.+0.j])
  作用在 |1⟩: tensor([0.+0.j, 1.+0.j])
  效果: |0⟩ → |0⟩, |1⟩ → |1⟩

X 门 (NOT门):
  作用在 |0⟩: tensor([0.+0.j, 1.+0.j])
  作用在 |1⟩: tensor([1.+0.j, 0.+0.j])
  效果: |0⟩ → |1⟩, |1⟩ → |0⟩

Y 门 (Y门):
  作用在 |0⟩: tensor([0.+0.j, 0.+1.j])
  作用在 |1⟩: tensor([0.-1.j, 0.+0.j])
  效果: |0⟩ → i|1⟩, |1⟩ → -i|0⟩

Z 门 (Z门):
  作用在 |0⟩: tensor([1.+0.j, 0.+0.j])
  作用在 |1⟩: tensor([ 0.+0.j, -1.+0.j])
  效果: |0⟩ → |0⟩, |1⟩ → -|1⟩


In [3]:
def tutorial_02_hadamard_gate():
    """Hadamard 门：创建叠加态"""
    print("\n" + "=" * 60)
    print("1.2 Hadamard 门 (H)")
    print("=" * 60)

    qdev = fq.DistributedQuantumDevice(n_wires=1, bsz=1, device="cpu")

    print("H 门矩阵: 1/√2 [[1, 1], [1, -1]]")

    qdev.reset_states()
    fq.H(wires=[0])(qdev)
    states = torch.view_as_complex(qdev.states)

    print(f"\nH|0⟩ = {states.flatten()}")
    print("  解释: (|0⟩ + |1⟩)/√2")

    qdev.reset_states()
    fq.X(wires=[0])(qdev)  # 变成 |1⟩
    fq.H(wires=[0])(qdev)
    states = torch.view_as_complex(qdev.states)

    print(f"\nH|1⟩ = {states.flatten()}")
    print("  解释: (|0⟩ - |1⟩)/√2")

In [4]:
tutorial_02_hadamard_gate()


1.2 Hadamard 门 (H)
H 门矩阵: 1/√2 [[1, 1], [1, -1]]

H|0⟩ = tensor([0.7071+0.j, 0.7071+0.j])
  解释: (|0⟩ + |1⟩)/√2

H|1⟩ = tensor([ 0.7071+0.j, -0.7071+0.j])
  解释: (|0⟩ - |1⟩)/√2


In [5]:
def tutorial_03_phase_gates():
    """相位门：S, T, P"""
    print("\n" + "=" * 60)
    print("1.3 相位门")
    print("=" * 60)

    qdev = fq.DistributedQuantumDevice(n_wires=1, bsz=1, device="cpu")

    # 先创建叠加态
    fq.H(wires=[0])(qdev)

    print("初始 |+⟩ 态:", torch.view_as_complex(qdev.states).flatten())
    states = torch.view_as_complex(qdev.states)
    angle = torch.atan2(states[0][1].imag, states[0][1].real)
    print(f"  |1⟩ 相位: 0 = {angle}")

    # S 门 (相位 π/2)
    qdev_s = fq.DistributedQuantumDevice(n_wires=1, bsz=1, device="cpu")
    fq.H(wires=[0])(qdev_s)
    fq.S(wires=[0])(qdev_s)
    states_s = torch.view_as_complex(qdev_s.states)
    angle_s = torch.atan2(states_s[0][1].imag, states_s[0][1].real)
    print(f"\nS 门 (π/2): {states_s.flatten()}")
    print(f"  |1⟩ 相位: π/2 = {angle_s}")

    # T 门 (相位 π/4)
    qdev_t = fq.DistributedQuantumDevice(n_wires=1, bsz=1, device="cpu")
    fq.H(wires=[0])(qdev_t)
    fq.T(wires=[0])(qdev_t)
    states_t = torch.view_as_complex(qdev_t.states)
    angle_t = torch.atan2(states_t[0][1].imag, states_t[0][1].real)
    print(f"\nT 门 (π/4): {states_t.flatten()}")
    print(f"  |1⟩ 相位: π/4 = {angle_t}")

In [6]:
tutorial_03_phase_gates()


1.3 相位门
初始 |+⟩ 态: tensor([0.7071+0.j, 0.7071+0.j])
  |1⟩ 相位: 0 = 0.0

S 门 (π/2): tensor([0.7071+0.0000j, 0.0000+0.7071j])
  |1⟩ 相位: π/2 = 1.5707963705062866

T 门 (π/4): tensor([0.7071+0.0000j, 0.5000+0.5000j])
  |1⟩ 相位: π/4 = 0.7853981852531433


In [ ]:
def tutorial_04_rotation_gates():
    """旋转门：RX, RY, RZ"""
    print("\n" + "=" * 60)
    print("1.4 旋转门 (RX, RY, RZ)")
    print("=" * 60)

    qdev = fq.DistributedQuantumDevice(n_wires=1, bsz=1, device="cpu")

    # 明确设置 theta = π/2
    theta = torch.tensor([torch.pi / 2])

    print(f"旋转角度 θ = π/2 = {theta.item():.4f} rad = 90°")
    print(f"  cos(θ/2) = cos({theta.item()/2:.4f}) = {torch.cos(theta/2).item():.4f}")
    print(f"  sin(θ/2) = sin({theta.item()/2:.4f}) = {torch.sin(theta/2).item():.4f}")

    # RX 门
    qdev.reset_states()
    fq.RX(wires=[0])(qdev, params=theta)
    states = torch.view_as_complex(qdev.states)

    amp_0 = states[0][0]
    amp_1 = states[0][1]
    phase_1 = torch.atan2(amp_1.imag, amp_1.real)

    print(f"\nRX(π/2)|0⟩ = {states.flatten()}")
    print(f"  |0⟩ 振幅: {amp_0:.4f}")
    print(f"  |1⟩ 振幅: {amp_1:.4f}")
    print(f"  |1⟩ 相位: {phase_1.item():.4f} rad = {phase_1.item() / torch.pi:.2f}π")

    # 验证
    print("  预期: 0.7071|0⟩ - 0.7071i|1⟩")
    print(f"  匹配: {abs(amp_0.item() - 0.7071) < 0.01 and abs(amp_1.item() + 0.7071*1j) < 0.01}")

    # RY 门
    qdev.reset_states()
    fq.RY(wires=[0])(qdev, params=theta)
    states = torch.view_as_complex(qdev.states)

    amp_0 = states[0][0]
    amp_1 = states[0][1]

    print(f"\nRY(π/2)|0⟩ = {states.flatten()}")
    print(f"  |0⟩ 振幅: {amp_0:.4f}")
    print(f"  |1⟩ 振幅: {amp_1:.4f}")

    # 验证
    print("  预期: 0.7071|0⟩ + 0.7071|1⟩")
    print(f"  匹配: {abs(amp_0.item() - 0.7071) < 0.01 and abs(amp_1.item() - 0.7071) < 0.01}")

    # RZ 门
    qdev.reset_states()
    fq.H(wires=[0])(qdev)
    print(f"\n初始 |+⟩ 态: {torch.view_as_complex(qdev.states).flatten()}")

    fq.RZ(wires=[0])(qdev, params=theta)
    states = torch.view_as_complex(qdev.states)

    amp_0 = states[0][0]
    amp_1 = states[0][1]
    phase_0 = torch.atan2(amp_0.imag, amp_0.real)
    phase_1 = torch.atan2(amp_1.imag, amp_1.real)
    relative_phase = phase_1 - phase_0

    print(f"\nRZ(π/2)|+⟩ = {states.flatten()}")
    print(f"  |0⟩ 相位: {phase_0.item():.4f} rad")
    print(f"  |1⟩ 相位: {phase_1.item():.4f} rad")
    print(f"  相对相位: {relative_phase.item():.4f} rad = {relative_phase.item() / torch.pi:.2f}π")
    print("  预期相对相位: π/2 = 1.5708 rad = 0.50π")
    print(f"  匹配: {abs(relative_phase.item() - 1.5708) < 0.01}")

In [8]:
tutorial_04_rotation_gates()


1.4 旋转门 (RX, RY, RZ)
旋转角度 θ = π/2 = 1.5708 rad = 90°
  cos(θ/2) = cos(0.7854) = 0.7071
  sin(θ/2) = sin(0.7854) = 0.7071

RX(π/2)|0⟩ = tensor([0.7071+0.0000j, 0.0000-0.7071j])
  |0⟩ 振幅: 0.7071+0.0000j
  |1⟩ 振幅: 0.0000-0.7071j
  |1⟩ 相位: -1.5708 rad = -0.50π
  预期: 0.7071|0⟩ - 0.7071i|1⟩
  匹配: True

RY(π/2)|0⟩ = tensor([0.7071+0.j, 0.7071+0.j])
  |0⟩ 振幅: 0.7071+0.0000j
  |1⟩ 振幅: 0.7071+0.0000j
  预期: 0.7071|0⟩ + 0.7071|1⟩
  匹配: True

初始 |+⟩ 态: tensor([0.7071+0.j, 0.7071+0.j])

RZ(π/2)|+⟩ = tensor([0.5000-0.5000j, 0.5000+0.5000j])
  |0⟩ 相位: -0.7854 rad
  |1⟩ 相位: 0.7854 rad
  相对相位: 1.5708 rad = 0.50π
  预期相对相位: π/2 = 1.5708 rad = 0.50π
  匹配: True


In [9]:
def tutorial_05_swap_gate():
    """SWAP 门：交换两个量子比特"""
    print("\n" + "=" * 60)
    print("1.5 SWAP 门")
    print("=" * 60)

    qdev = fq.DistributedQuantumDevice(n_wires=2, bsz=1, device="cpu")

    # 创建 |10⟩ 态 (q0=1, q1=0)
    fq.X(wires=[0])(qdev)

    print("初始状态: |10⟩ (q0=1, q1=0)")
    states = torch.view_as_complex(qdev.states)
    states_flat = states.flatten()
    print(f"  状态向量: {states_flat}")

    # 找出非零振幅
    for i, amp in enumerate(states_flat):
        if abs(amp) > 0.5:
            binary = format(i, f'0{qdev.n_wires}b')
            print(f"  解释: |{binary}⟩ 振幅 = {amp:.4f}")

    # SWAP 门
    fq.SWAP(wires=[0, 1])(qdev)

    print("\nSWAP(0,1) 后: |01⟩ (q0=0, q1=1)")
    states = torch.view_as_complex(qdev.states)
    states_flat = states.flatten()
    print(f"  状态向量: {states_flat}")

    for i, amp in enumerate(states_flat):
        if abs(amp) > 0.5:
            binary = format(i, f'0{qdev.n_wires}b')
            print(f"  解释: |{binary}⟩ 振幅 = {amp:.4f}")

    print("\n  结论: SWAP 门交换了两个量子比特的状态")

In [10]:
tutorial_05_swap_gate()


1.5 SWAP 门
初始状态: |10⟩ (q0=1, q1=0)
  状态向量: tensor([0.+0.j, 0.+0.j, 1.+0.j, 0.+0.j])
  解释: |10⟩ 振幅 = 1.0000+0.0000j

SWAP(0,1) 后: |01⟩ (q0=0, q1=1)
  状态向量: tensor([0.+0.j, 1.+0.j, 0.+0.j, 0.+0.j])
  解释: |01⟩ 振幅 = 1.0000+0.0000j

  结论: SWAP 门交换了两个量子比特的状态
